## 1. Imports and Configuration

In [1]:
import os
import re
import numpy as np
import nibabel as nib
#  
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

ModuleNotFoundError: No module named 'ipywidgets'

In [ ]:
# Configuration
# Point this to where your source (input) modality scans are stored
source_dir = "/storage/an_inam/MR2MR/Data/10_Pat_t1/3T_7TReg" 

comparison_directories = [
    {
        "model_tag" : "3DEU",
        "logs_dir" : "logs/VNET_L1_DS2_10fold/generations" 
    }
    # Add more models here...
]

def extract_subject_id(filename):
    """Extracts the sub-* identifier from a filename."""
    match = re.search(r'(sub-[a-zA-Z0-9]+)', filename)
    if match:
        return match.group(1)
    else:
        raise ValueError(f"Could not find 'sub-*' in filename: {filename}")

## 2. Data Loading and Affine Alignment

In [ ]:
def load_subject_scans(subject_id, enforce_orig_affine=True):
    """
    Loads Source, Orig (Ground Truth), and Fake (Model) scans for a given subject.
    Raises FileNotFoundError if any scan is missing.
    """
    scans_dict = {}
    
    # 1. Find and load Source (Ground)
    # Assuming the source file has the subject ID in it. Adjust the glob/search if needed.
    source_files = [f for f in os.listdir(source_dir) if subject_id in f]
    if not source_files:
        raise FileNotFoundError(f"Missing Source (Ground) scan for {subject_id} in {source_dir}")
    source_path = os.path.join(source_dir, source_files[0])
    scans_dict['Source'] = nib.load(source_path)
    
    # 2. Find and load Orig (Ground Truth)
    # We grab this from the first model's directory as a reference
    ref_dir = comparison_directories[0]['logs_dir']
    orig_path = os.path.join(ref_dir, f"{subject_id}_orig.nii.gz") # Adjust suffix if needed
    if not os.path.exists(orig_path):
        raise FileNotFoundError(f"Missing Orig (Ground Truth) scan: {orig_path}")
    
    orig_img = nib.load(orig_path)
    scans_dict['Orig'] = orig_img
    
    # 3. Load Model Outputs (Fakes)
    for model in comparison_directories:
        tag = model['model_tag']
        fake_path = os.path.join(model['logs_dir'], f"{subject_id}_fake.nii.gz") # Adjust suffix
        
        if not os.path.exists(fake_path):
            raise FileNotFoundError(f"Missing Fake scan for model {tag}: {fake_path}")
            
        fake_img = nib.load(fake_path)
        
        # Enforce Affine if requested
        if enforce_orig_affine:
            # Nilearn resamples the fake image to strictly match the orig image's affine and shape
            fake_img = nilearn.image.resample_to_img(fake_img, orig_img, interpolation='continuous')
            
        scans_dict[tag] = fake_img
        
    # If enforcing affine, also align the source to the orig for consistent viewing
    if enforce_orig_affine:
         scans_dict['Source'] = nilearn.image.resample_to_img(scans_dict['Source'], orig_img, interpolation='continuous')
         
    return scans_dict

# Extract available subjects from the first model's directory
sample_dir = comparison_directories[0]['logs_dir']
available_files = [f for f in os.listdir(sample_dir) if "_orig" in f]
available_subjects = sorted(list(set([extract_subject_id(f) for f in available_files])))
print(f"Found {len(available_subjects)} complete subjects.")